##### CHROMADB

In [1]:
# Data Loading

from langchain_community.document_loaders import TextLoader

loader=TextLoader("speech.txt") # just initializing the loader object

data=loader.load()

C:\Users\Azrap\AppData\Local\Temp\ipykernel_17804\593643623.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
e:\genai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# split the data into chunks using recursive or character split

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splitted_text=splitter.split_documents(data)

In [3]:
splitted_text

[Document(metadata={'source': 'speech.txt'}, page_content="The Clockmaker of Ratnapur\n\nIn a small hill town called Ratnapur, there lived an old clockmaker named Dinkar Rao. His shop stood at the end of Lantern Street, right next to the town's only bakery. The shop was tiny, but it was packed with hundreds of clocks: wall clocks, pocket watches, cuckoo clocks, and even a giant grandfather clock that had not worked for forty years."),
 Document(metadata={'source': 'speech.txt'}, page_content='Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."'),
 Document(metadata={'source': 'speech.txt'}, page_conten

In [4]:
# after splitting the data into chunks next is to create vectors 
# means emebdiing and then storing the details into vectore store ->faiss or chroma 

from langchain_community.embeddings import OllamaEmbeddings

embeddings=OllamaEmbeddings(
    model="nomic-embed-text"
)

C:\Users\Azrap\AppData\Local\Temp\ipykernel_17804\3146289032.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings=OllamaEmbeddings(


In [5]:
from langchain_chroma import Chroma

vector_store=Chroma.from_documents(
    documents=splitted_text,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [6]:
query = "Why did Dinkar keep the old grandfather clock stopped for so many years?"

results = vector_store.similarity_search(query) # this finds only the similarr pieces of chunks regarding the query

results

[Document(id='507833bb-b8a2-4dfd-9361-d2a8d3f039aa', metadata={'source': 'speech.txt'}, page_content='Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."'),
 Document(id='2df85933-9610-4829-b060-2b587115a5d7', metadata={'source': 'speech.txt'}, page_content='"I always thought he was angry with me," Dinkar said quietly. "I stopped the clock that night, and I never started it again. I promised myself I would only wind it when Harish returned."\n\nMeera looked at the numbers on the paper. "Seven, twenty-one, nineteen seventy-four. It is the date you finished the clock. But why would my grandfather hide it

In [7]:
res_with_scores=vector_store.similarity_search_with_score(query)

res_with_scores

[(Document(id='507833bb-b8a2-4dfd-9361-d2a8d3f039aa', metadata={'source': 'speech.txt'}, page_content='Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."'),
  219.03311157226562),
 (Document(id='2df85933-9610-4829-b060-2b587115a5d7', metadata={'source': 'speech.txt'}, page_content='"I always thought he was angry with me," Dinkar said quietly. "I stopped the clock that night, and I never started it again. I promised myself I would only wind it when Harish returned."\n\nMeera looked at the numbers on the paper. "Seven, twenty-one, nineteen seventy-four. It is the date you finished the clock. But why wou

In [8]:
# retiever 

retriever=vector_store.as_retriever()

retriever.invoke(query)

[Document(id='507833bb-b8a2-4dfd-9361-d2a8d3f039aa', metadata={'source': 'speech.txt'}, page_content='Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."'),
 Document(id='2df85933-9610-4829-b060-2b587115a5d7', metadata={'source': 'speech.txt'}, page_content='"I always thought he was angry with me," Dinkar said quietly. "I stopped the clock that night, and I never started it again. I promised myself I would only wind it when Harish returned."\n\nMeera looked at the numbers on the paper. "Seven, twenty-one, nineteen seventy-four. It is the date you finished the clock. But why would my grandfather hide it